# ICC profiles

Open, inspect, edit, create and save ICC profiles.

In [1]:
import os
import tempfile
from pathlib import Path

import numpy as np

import lcms2

out = Path(os.environ.get("LCMS2_EXAMPLES_OUT") or tempfile.mkdtemp())

`Profile(name)` opens a built-in profile; names are case-insensitive. `Profile.builtins()` lists them.

In [2]:
srgb = lcms2.Profile("sRGB")
print(len(lcms2.Profile.builtins()), "built-ins")
print(srgb)

25 built-ins
<Profile 'sRGB built-in' RGB display v4.4>


The header and the decoded tags are properties.

In [3]:
print(srgb.color_space, srgb.pcs, srgb.device_class, srgb.channels, srgb.version)
print(srgb.tags)
print(srgb.is_matrix_shaper, [intent.name for intent in srgb.supported_intents])
print(srgb.media_white_point.round(4))

RGB XYZ display 3 4.4
['desc', 'cprt', 'wtpt', 'chad', 'rXYZ', 'bXYZ', 'gXYZ', 'rTRC', 'gTRC', 'bTRC', 'chrm']
True ['PERCEPTUAL', 'RELATIVE_COLORIMETRIC', 'SATURATION', 'ABSOLUTE_COLORIMETRIC']
[0.9642 1.     0.8249]


Matrix/TRC profiles expose their colorants, one XYZ row per channel, and their tone curves.

In [4]:
print(srgb.colorants.round(4))
print(srgb.trc["rTRC"])
print(srgb.trc["rTRC"]([0.0, 0.5, 1.0]).round(4))

[[0.436  0.2225 0.0139]
 [0.3851 0.7169 0.0971]
 [0.143  0.0606 0.7139]]
ToneCurve(parametric type 4, 4096 entries, gamma~2.2)
[0.    0.214 1.   ]


Text fields are settable. `save` writes a file and `Profile.open` reads it back; `to_bytes` and
`from_bytes` do the same in memory, for example for a profile embedded in an image.

In [5]:
custom = lcms2.Profile("Display P3")
custom.description = "My display"
custom.copyright = "Public domain"
custom.compute_id()
path = out / "my_display.icc"
custom.save(path)
print(path.name, path.stat().st_size, "bytes")

reopened = lcms2.Profile.open(path)
print(reopened.description, reopened.profile_id == custom.profile_id)
print(lcms2.Profile.from_bytes(custom.to_bytes()).description)

my_display.icc 560 bytes
My display True
My display


Create profiles from numbers: primaries and a gamma for RGB, or a white point and gamma for gray.
White points are illuminant names, a `temperature=`, or xyY.

In [6]:
wide = lcms2.Profile.rgb("D65", ((0.708, 0.292), (0.170, 0.797), (0.131, 0.046)), 2.2, description="Wide RGB")
gray = lcms2.Profile.gray("D65", 2.2)
print(wide, gray)
print(wide.colorants.round(4))

<Profile 'Wide RGB' RGB display v4.4> <Profile 'gray built-in' GRAY display v4.4>
[[ 0.6735  0.279  -0.0019]
 [ 0.1657  0.6753  0.03  ]
 [ 0.1251  0.0456  0.7969]]


`Profile.transform` converts pixels from this profile to another profile or color space name.

In [7]:
rgb = np.array([[255, 128, 0], [10, 200, 30]], np.uint8)
print(wide.transform(rgb, "sRGB"))
print(srgb.transform(rgb, "Lab").round(2))

[[255  99   0]
 [  0 213   0]]
[[ 67.82  45.49  74.84]
 [ 70.56 -65.    63.34]]


The Lab and XYZ virtual profiles and the sRGB one are also constructors.

In [8]:
print(lcms2.Profile.lab(white_point="D65", version=4))
print(lcms2.Profile.xyz())
print(lcms2.Profile.srgb().description)

<Profile 'Lab identity built-in' Lab abstract v4.4>
<Profile 'XYZ identity built-in' XYZ abstract v4.4>
sRGB built-in
